# 05 学生向け研究実験テンプレート

## 1. 研究目的

この Notebook は、チームで研究課題を選び、**自分たちで実験を追加する**ためのテンプレートです。
**完成した研究結果は入っていません。** 各 Research の「TODO」を埋めて、実験・結果・考察を記録してください。

| Research | テーマ | 関連する Notebook |
| --- | --- | --- |
| A | Spike timing と情報表現（ノイズへの頑健性） | 02 |
| B | ANN と LUT の演算量 | 03 |
| C | CPU・GPU・TPU | 04 |
| D | LUT のメモリ使用量 | 03 |
| E | 簡易分類課題（ANN と LUT 分類器） | 01〜03 |

少なくとも 1 つを選んでください（README の Core Research・Advanced Research も参照）。

## 2. 理論的背景

Notebook 01〜04 と README（6〜10 章、17 章）を参照してください。ここでは研究の進め方の注意だけをまとめます。

- **仮説を先に書く**：実験の前に「こうなるはず」と理由つきで書く。
- **条件を記録する**：問題規模（$n, m, n_t, n_c, T, B$）、乱数シード、実行環境、測定回数と統計量（中央値など）。
- **3 つの比較を区別する**：① 計算原理（演算回数）、② 実装の処理時間、③ 同一タスクでの性能（精度）。
- **測っていないことを結論しない**：消費電力、ニューロモルフィックハードウェアでの性能など。
- **論文のモデルと教育用簡略モデルを区別する**：この教材の LUT は学習なし・小規模です。

## 3. 実験環境（最初に必ず実行）

このセルは、E3 リポジトリの `.py` ファイル（`ann.py`、`spike_lut.py` など）を import できるように、
**作業ディレクトリ（E3 のルート）を自動で探して設定**します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのチームのリポジトリの URL** を入れる（`REPO_SUBDIR` に E3 のフォルダ名） |
| Colab（方法B：ZIP をアップロード） | E3 フォルダを ZIP にして、Colab の左の「ファイル」に `/content` へアップロードしておく（またはこのセルの実行中にアップロード） |

> ⚠️ `textbook/` フォルダ（教科書のコード）は**公開リポジトリには含めない**約束です。
> 方法A で `textbook/` がない場合、`spike_encoding.py`・`benchmark.py` の import が失敗します。
> その場合は方法B（ZIP）を使うか、授業で配布された `textbook/` を同じ場所に置いてください。

In [ ]:
import os
import sys
import glob
import zipfile
import subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                           # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "e3-ann-snn-comparison"   # リポジトリの中の E3 フォルダ（ルートが E3 なら "" にする）

IN_COLAB = "google.colab" in sys.modules


def is_e3_root(p):
    """spike_lut.py と ann.py があるフォルダを E3 のルートとみなす"""
    return (Path(p) / "spike_lut.py").exists() and (Path(p) / "ann.py").exists()


def find_e3_root():
    # 1) ローカル：notebooks/ の中、または E3 のルートで開いた場合
    for p in [Path.cwd(), Path.cwd().parent]:
        if is_e3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    # 2) Colab 方法A：clone
    if REPO_URL:
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_e3_root(cand):
            return cand
    # 3) Colab 方法B：/content にある ZIP を展開（なければアップロードを求める）
    zips = glob.glob("/content/*.zip")
    if not zips:
        from google.colab import files
        print("E3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/e3_upload")
    for hit in glob.glob("/content/e3_upload/**/spike_lut.py", recursive=True):
        if is_e3_root(Path(hit).parent):
            return Path(hit).parent
    return None


E3_ROOT = find_e3_root()
if E3_ROOT is None:
    raise RuntimeError("E3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(E3_ROOT)
if str(E3_ROOT) not in sys.path:
    sys.path.insert(0, str(E3_ROOT))
print("E3_ROOT:", E3_ROOT)
if not (E3_ROOT / "textbook" / "insertion_sort.py").exists():
    print("⚠️ textbook/ がありません。spike_encoding.py などの import が失敗します（上の注意を参照）。")

In [ ]:
import matplotlib.pyplot as plt

# グラフの共通スタイル（Colab には日本語フォントがないことが多いので、図の文字は英語にしている）
plt.rcParams.update({
    "figure.figsize": (7, 3.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})
# 色は役割ごとに固定する（ANN / LIF・SNN / LUT、CPU / GPU / TPU）
COLORS = {"ANN": "#2a78d6", "LIF": "#1baf7a", "LUT": "#eb6834",
          "CPU": "#2a78d6", "GPU": "#eb6834", "TPU": "#1baf7a", "neutral": "#8a8985"}

## 4. 使用するコード

既存の `.py` をそのまま import します。新しい関数は**この Notebook の中**に書き、よくできたものは `.py` に移してテストを追加してください。

In [ ]:
import json
import math
import random
import itertools
import numpy as np

from ann import dense_forward, ann_operation_counts
from lif_snn import simulate_lif, latency_encode
from spike_encoding import spike_order, spike_counts, num_orders, quantize
from spike_lut import (SpikeLUT, compare_pairs, bits_to_index, lut_forward,
                       make_random_tables, lut_operation_counts)
from benchmark import make_ann, traced_memory
import tensor_backends as tb

SEED = 0   # TODO：使った乱数シードを報告に書く
with open("data/sample_patterns.json", encoding="utf-8") as f:
    data = json.load(f)

## 5. 実行例・6. 可視化・7. 解釈 ―― 各 Research で自分たちで行う

以下の各 Research には、**最低限の材料（関数・変数）**と **TODO** だけを置いています。
セルはそのままでもエラーなく実行できますが、結果は何も出しません。

---
## Research A：Spike timing と情報表現

**目的**：発火順序の変更やノイズによって LUT index がどれだけ変わるかを測り、spike timing による情報表現の頑健性を評価する。

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | 発火時刻、ノイズの大きさ σ、時刻の分解能、anchor pair の選び方 |
| 評価指標 | 発火順序が変わる割合、LUT index が変わる割合、到達できる index の数 |
| 固定する条件・乱数シード | TODO |
| 担当者 | TODO |

In [ ]:
def add_noise(latencies, sigma, rng):
    """発火時刻にガウスノイズを加える（rng = random.Random(seed)）"""
    return [t + rng.gauss(0, sigma) for t in latencies]


base = data["latency_example"]
tables_A = make_random_tables(n_inputs=len(base), n_tables=3, n_comparisons=3, out_dim=4, seed=SEED)
sigmas = [0.0, 0.01, 0.02, 0.05, 0.1]   # TODO：範囲を決める

# TODO A-1：発火順序を変更したパターン（例：pattern_a と pattern_b）で、index がどう変わるか調べる
# TODO A-2：sigma ごとに、ノイズを加えたときに「発火順序が変わる割合」を測る
# TODO A-3：sigma ごとに、「LUT index が 1 つでも変わる割合」を測る
# TODO A-4：比較する 2 つの発火時刻の差が小さい anchor pair ほど変わりやすいか調べる
# TODO A-5：quantize() で分解能を粗くしたとき、同時発火と index がどう変わるか調べる
results_A = []   # 例：{"sigma": ..., "order_change_rate": ..., "index_change_rate": ...}

**結果**（表・グラフを貼る）：TODO

**考察**（仮説は正しかったか、なぜそうなったか、論文の主張とどう対応するか）：TODO

**限界**（この実験で言えないこと）：TODO

---
## Research B：ANN と LUT の演算量

**目的**：入力次元・出力次元・LUT 数・anchor pair 数を変えたときの演算量を、理論式と実測で比較する。

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | n、m、nt、nc |
| 評価指標 | 乗算・加算・比較・LUT 参照の回数、保存容量、読み出し量、実行時間（中央値） |
| 固定する条件・乱数シード | TODO |
| 担当者 | TODO |

In [ ]:
conditions_B = [
    # (n, m, nt, nc)  TODO：調べたい条件を追加する
    (16, 16, 4, 4),
    (64, 64, 16, 6),
]
for n, m, nt, nc in conditions_B:
    print((n, m, nt, nc), "ANN:", ann_operation_counts(n, m))
    print((n, m, nt, nc), "LUT:", lut_operation_counts(nt, nc, m))

# TODO B-1：条件を系統的に変えた表を作る（どれか 1 つだけを変え、他は固定する）
# TODO B-2：tb.time_call を使って実行時間を測り、演算回数と比べる
# TODO B-3：ANN と LUT が「同じくらいの演算量」になる条件を探し、そのときの保存容量を比べる
# TODO B-4：グラフ化（横軸：操作変数、縦軸：演算回数や時間。対数軸が必要か考える）
results_B = []

**結果**（表・グラフを貼る）：TODO

**考察**（仮説は正しかったか、なぜそうなったか、論文の主張とどう対応するか）：TODO

**限界**（この実験で言えないこと）：TODO

---
## Research C：CPU・GPU・TPU

**目的**：バッチサイズを変えて演算時間を測り、デバイスによる性能差と、データ転送・コンパイルの影響を分析する。

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | デバイス（ランタイム）、バッチサイズ B、問題規模、dtype |
| 評価指標 | warm median 時間、初回時間、コンパイル時間、転送時間 |
| 固定する条件・乱数シード | TODO |
| 担当者 | TODO |

Research C の測定には **Notebook 04** を使ってください（ランタイムごとに `notebooks/results/04_benchmark_<cpu|gpu|tpu>.csv` が保存されます）。
ここでは保存した CSV を読み込んで分析します。

> 使えなかったデバイスの値を推測で埋めないでください。TPU が使えなかった場合は「使えなかった」と記録します。

In [ ]:
import csv, glob, os

rows_C = []
for path in sorted(glob.glob("notebooks/results/04_benchmark_*.csv")):
    with open(path) as f:
        rows_C += [dict(r, runtime=os.path.basename(path)) for r in csv.DictReader(f)]
print(len(rows_C), "rows from", sorted({r["runtime"] for r in rows_C}) or "no CSV yet (run notebook 04 first)")

# TODO C-1：バッチサイズ B と warm_median_s の関係を、デバイスごとにグラフにする
# TODO C-2：GPU（または TPU）が CPU（NumPy）より速くなる B の境目を求める
# TODO C-3：input_transfer_s + warm_median_s（毎回転送する場合）で比べ直す
# TODO C-4：JAX の compile_s が何回分の計算に相当するか計算する
# TODO C-5：ANN と LUT で、デバイスによる速さの違いが同じか比べる

**結果**（表・グラフを貼る）：TODO

**考察**（仮説は正しかったか、なぜそうなったか、論文の主張とどう対応するか）：TODO

**限界**（この実験で言えないこと）：TODO

---
## Research D：LUT のメモリ使用量

**目的**：LUT の行数・パラメータ数・メモリ使用量を測り、anchor pair 数とメモリ容量の関係を分析する。

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | nc（行数 2^nc）、nt、m、dtype |
| 評価指標 | パラメータ数（理論値）、tracemalloc の測定値、NumPy 配列の nbytes、1 回の forward で読む値の数 |
| 固定する条件・乱数シード | TODO |
| 担当者 | TODO |

In [ ]:
def lut_memory(nt, nc, m, seed=SEED):
    """理論値と測定値を返す（測定は Python のリストと NumPy 配列の両方）"""
    params = lut_operation_counts(nt, nc, m)["parameters"]
    traced = traced_memory(lambda: make_random_tables(m, nt, nc, m, seed=seed))
    _, rows = tb.tables_to_arrays(make_random_tables(m, nt, nc, m, seed=seed))
    return {"nt": nt, "nc": nc, "m": m, "params": params,
            "theory_float32_bytes": 4 * params, "python_bytes": traced, "numpy_bytes": rows.nbytes}

print(lut_memory(nt=4, nc=4, m=8))   # 例：1 条件だけ

# TODO D-1：nc を 1 ずつ増やして（大きくしすぎない：nc <= 14 程度）、行数とメモリの関係を表にする
# TODO D-2：python_bytes と numpy_bytes と理論値の違いを説明する
# TODO D-3：保存容量（params）と 1 回の読み出し量（values_read）の比をグラフにする
# TODO D-4：同じパラメータ数の ANN（n*m + m = params になる n, m）と比べる
results_D = []

**結果**（表・グラフを貼る）：TODO

**考察**（仮説は正しかったか、なぜそうなったか、論文の主張とどう対応するか）：TODO

**限界**（この実験で言えないこと）：TODO

---
## Research E：簡易分類課題

**目的**：人工的な分類データを作り、ANN と LUT を使った分類器の精度・モデル容量・計算コストを比較する。

| 項目 | 記入欄 |
| --- | --- |
| 仮説（実験の前に書く） | TODO |
| 操作変数 | クラス数、ノイズの大きさ、モデルの大きさ（ANN の隠れ層、LUT の nt・nc） |
| 評価指標 | テスト精度、パラメータ数、forward の演算回数・時間、学習時間 |
| 固定する条件・乱数シード | TODO |
| 担当者 | TODO |

**人工データの例**：各クラスに「典型的な発火順序（プロトタイプの latency vector）」を 1 つ決め、それにノイズを加えたサンプルを作ります。
データ生成と精度計算の関数だけを用意しています。**分類器の学習は TODO です。**

> ⚠️ 自分たちで考えた学習則と、論文の surrogate gradient による誤差逆伝播（Section III）は**別物**です。報告では区別してください。
> ANN と LUT でモデルの容量（パラメータ数）が違う場合は、それを明記してください。

In [ ]:
def make_order_dataset(n_classes, n_neurons, n_per_class, sigma, seed=SEED):
    """クラスごとのプロトタイプ latency vector にノイズを加えたデータ (X, y) を作る"""
    rng = random.Random(seed)
    prototypes = [[rng.random() for _ in range(n_neurons)] for _ in range(n_classes)]
    X, y = [], []
    for c, proto in enumerate(prototypes):
        for _ in range(n_per_class):
            X.append([t + rng.gauss(0, sigma) for t in proto])
            y.append(c)
    idx = list(range(len(X)))
    rng.shuffle(idx)
    return [X[i] for i in idx], [y[i] for i in idx]


def accuracy(predict, X, y):
    """predict(x) がクラス番号を返す関数のとき、正解率を返す"""
    return sum(predict(x) == t for x, t in zip(X, y)) / len(y)


X_all, y_all = make_order_dataset(n_classes=3, n_neurons=8, n_per_class=100, sigma=0.05)
split = int(0.8 * len(X_all))
X_train, y_train, X_test, y_test = X_all[:split], y_all[:split], X_all[split:], y_all[split:]
print("train:", len(X_train), " test:", len(X_test), " classes:", sorted(set(y_all)))

# 参考：常に 0 と答える分類器の精度（これより良くなければ意味がない）
print("baseline (always class 0):", accuracy(lambda x: 0, X_test, y_test))

In [ ]:
# ---- ANN 分類器（TODO）----
# 例：出力 m = クラス数 の全結合層。予測 = 出力が最大のクラス
n_in, n_cls = 8, 3
W_E, b_E = make_ann(n_in, n_cls, SEED)       # 学習前の乱数の重み

def ann_predict(x):
    y = dense_forward(x, W_E, b_E)
    return max(range(len(y)), key=lambda k: y[k])

# TODO E-1：W_E, b_E を学習する（例：NumPy で softmax + 交差エントロピーの勾配降下、など）
print("ANN (untrained) test accuracy:", accuracy(ann_predict, X_test, y_test))

In [ ]:
# ---- LUT 分類器（TODO）----
# 例：出力 m = クラス数 の LUT。予測 = 出力（選ばれた行の和）が最大のクラス
tables_E = make_random_tables(n_inputs=8, n_tables=4, n_comparisons=4, out_dim=3, seed=SEED)

def lut_predict(x):
    y = lut_forward(x, tables_E)["output"]
    return max(range(len(y)), key=lambda k: y[k])

# TODO E-2：LUT の行の値を学習する方法を考えて実装する
#           （例：正解クラスの成分を少し増やし、間違えたクラスの成分を少し減らす、など。
#             選ばれた行だけが更新されることに注意）
# TODO E-3：nt・nc を変えて、精度とパラメータ数（lut_operation_counts）の関係を調べる
# TODO E-4：ANN と LUT の精度・パラメータ数・forward 時間・学習時間を 1 つの表にまとめる
# TODO E-5：sigma（ノイズ）を大きくしたとき、どちらの精度がどう下がるか調べる
print("LUT (untrained) test accuracy:", accuracy(lut_predict, X_test, y_test))

**結果**（表・グラフを貼る）：TODO

**考察**（仮説は正しかったか、なぜそうなったか、論文の主張とどう対応するか）：TODO

**限界**（この実験で言えないこと）：TODO

## 8. 学生向け TODO（まとめ）

- [ ] 取り組む Research を選び、仮説・操作変数・評価指標・乱数シードを表に記入する
- [ ] 各 Research の TODO を実装する
- [ ] 結果を表・グラフにまとめ、`notebooks/results/` などに保存する
- [ ] 結果の解釈と限界を書く
- [ ] 再利用できる関数は `.py` に移し、`tests/` にテストを追加する（`python3 -m unittest discover -s tests -v` が通ること）

## 9. 考察課題

1. RQ1〜RQ4（README 1 章）に対して、自分たちの実験からどこまで答えられましたか。
2. 論文のどの部分を再現し、どの部分を再現していませんか（README 17 章の表を使う）。
3. 自分たちの結果のうち、「計算原理の比較」「実装の処理時間比較」「同一タスクでの性能比較」のどれに当たるかを分類してください。
4. この教材の簡略モデルを、論文のモデルに近づけるには何が必要ですか。

## 10. 参考文献

- E. Izhikevich, "Spiking Manifesto", arXiv:2512.11843, 2025. <https://arxiv.org/abs/2512.11843>（arXiv の表示ライセンス：CC BY-NC-SA 4.0）
- 著者の公開コード：<https://github.com/izhikevich/SNN>（この教材には**コピーしていません**）
- W. Gerstner, W. M. Kistler, R. Naud, L. Paninski, *Neuronal Dynamics*, Cambridge University Press, 2014（LIF モデル）
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 探索、第5章 再帰、第6章 ソート）
- E3 リポジトリの `README.md`（特に 17章「論文との対応関係」）

> この Notebook は論文の中心的なアイデアを**教育用に簡略化**したものです。論文のモデル・著者の実装そのものではありません。